In [2]:
import ccxt
import pandas as pd
import numpy as np
import ta

exchange = ccxt.binance({
    "enableRateLimit": True,
    "options": {
        "defaultType": "future"   # ← THIS switches to futures
    }
})



In [3]:
markets = exchange.load_markets()

usdt_pairs = [
    symbol for symbol, market in markets.items()
    if market['quote'] == 'USDT'     # Must be USDT pair
    and market['contract']           # Must be derivative
    and market['swap']               # Must be perpetual (not quarterly)
    and market['active']             # Must be active
]

print(f"Total active USDT futures pairs: {len(usdt_pairs)}")
print(usdt_pairs[:10])


Total active USDT futures pairs: 579
['BTC/USDT:USDT', 'ETH/USDT:USDT', 'BCH/USDT:USDT', 'XRP/USDT:USDT', 'LTC/USDT:USDT', 'TRX/USDT:USDT', 'ETC/USDT:USDT', 'LINK/USDT:USDT', 'XLM/USDT:USDT', 'ADA/USDT:USDT']


In [4]:
filtered_pairs = []

for symbol in usdt_pairs:
    print(f"scan {symbol}")
    ticker = exchange.fetch_ticker(symbol)

    if ticker['quoteVolume'] and ticker['quoteVolume'] > 5_000_000:
        filtered_pairs.append(symbol)

print(f"High-liquidity USDT futures: {len(filtered_pairs)}")


scan BTC/USDT:USDT
scan ETH/USDT:USDT
scan BCH/USDT:USDT
scan XRP/USDT:USDT
scan LTC/USDT:USDT
scan TRX/USDT:USDT
scan ETC/USDT:USDT
scan LINK/USDT:USDT
scan XLM/USDT:USDT
scan ADA/USDT:USDT
scan XMR/USDT:USDT
scan DASH/USDT:USDT
scan ZEC/USDT:USDT
scan XTZ/USDT:USDT
scan BNB/USDT:USDT
scan ATOM/USDT:USDT
scan ONT/USDT:USDT
scan IOTA/USDT:USDT
scan BAT/USDT:USDT
scan VET/USDT:USDT
scan NEO/USDT:USDT
scan QTUM/USDT:USDT
scan IOST/USDT:USDT
scan THETA/USDT:USDT
scan ALGO/USDT:USDT
scan ZIL/USDT:USDT
scan KNC/USDT:USDT
scan ZRX/USDT:USDT
scan COMP/USDT:USDT
scan DOGE/USDT:USDT
scan KAVA/USDT:USDT
scan BAND/USDT:USDT
scan RLC/USDT:USDT
scan SNX/USDT:USDT
scan DOT/USDT:USDT
scan YFI/USDT:USDT
scan CRV/USDT:USDT
scan TRB/USDT:USDT
scan RUNE/USDT:USDT
scan SUSHI/USDT:USDT
scan EGLD/USDT:USDT
scan SOL/USDT:USDT
scan ICX/USDT:USDT
scan STORJ/USDT:USDT
scan UNI/USDT:USDT
scan AVAX/USDT:USDT
scan ENJ/USDT:USDT
scan KSM/USDT:USDT
scan NEAR/USDT:USDT
scan AAVE/USDT:USDT
scan FIL/USDT:USDT
scan RSR/

In [5]:
def get_ohlcv(symbol, timeframe='15m', limit=200):
    ohlcv = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)

    df = pd.DataFrame(ohlcv, columns=[
        'timestamp','open','high','low','close','volume'
    ])

    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    return df


In [6]:
import pandas as pd
import numpy as np

def add_rsi_tradingview(df, length=14, source_col='close'):
    """
    TradingView / Pine Script RSI implementation using RMA (Wilder)
    """

    # 1. Price change (ta.change)
    change = df[source_col].diff()

    # 2. Up & Down series
    up = change.clip(lower=0)
    down = -change.clip(upper=0)

    # 3. Wilder's Moving Average (RMA)
    alpha = 1 / length
    avg_up = up.ewm(alpha=alpha, adjust=False).mean()
    avg_down = down.ewm(alpha=alpha, adjust=False).mean()

    # 4. RSI calculation
    rs = avg_up / avg_down
    rsi = np.where(
        avg_down == 0, 100,
        np.where(avg_up == 0, 0, 100 - (100 / (1 + rs)))
    )

    df['rsi'] = rsi
    return df


In [7]:
def add_stochastic_tv(df, periodK=14, smoothK=3, periodD=3):
    
    # Raw stochastic
    low_min = df['low'].rolling(periodK).min()
    high_max = df['high'].rolling(periodK).max()
    
    stoch = 100 * (df['close'] - low_min) / (high_max - low_min)
    
    # Smooth K
    k = stoch.rolling(smoothK).mean()
    
    # Smooth D
    d = k.rolling(periodD).mean()
    
    df['stoch_k'] = k
    df['stoch_d'] = d
    
    return df

In [8]:
def add_stoch_rsi_tv(df, lengthRSI=14, lengthStoch=14, smoothK=3, smoothD=3):

    # --- TRUE TradingView RSI (RMA based)
    delta = df['close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    alpha = 1 / lengthRSI
    avg_gain = gain.ewm(alpha=alpha, adjust=False).mean()
    avg_loss = loss.ewm(alpha=alpha, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    # --- TRUE Stochastic of RSI
    rsi_min = rsi.rolling(lengthStoch).min()
    rsi_max = rsi.rolling(lengthStoch).max()

    stoch = 100 * (rsi - rsi_min) / (rsi_max - rsi_min)

    # --- Smooth K and D
    k = stoch.rolling(smoothK).mean()
    d = k.rolling(smoothD).mean()

    df['stoch_rsi_k'] = k
    df['stoch_rsi_d'] = d

    return df

In [9]:
def check_signal(df):
    last = df.iloc[-1]

    rsi = last["rsi"]
    stoch_rsi = last["stoch_rsi_k"]
    stoch_k = last["stoch_d"]

    # =========================
    # LONG CONDITIONS
    # =========================
    oversold = (
        rsi < 45 and
        stoch_rsi < 45 and
        stoch_k < 45
    )

    momentum_up = stoch_k > stoch_rsi

    long_signal = oversold and momentum_up

    # =========================
    # SHORT CONDITIONS (OPPOSITE)
    # =========================
    overbought = (
        rsi > 55 and
        stoch_rsi > 55 and
        stoch_k > 55
    )

    momentum_down = stoch_k < stoch_rsi

    short_signal = overbought and momentum_down

    if long_signal:
        return "LONG"
    elif short_signal:
        return "SHORT"
    else:
        return None


In [10]:
timeframes = ['15m', '1h', '4h', '1d']

def analyze_symbol(symbol):
    long_agree = []
    short_agree = []

    for tf in timeframes:
        try:
            df = get_ohlcv(symbol, tf, 500)
            df = df.iloc[:-1]

            df = add_rsi_tradingview(df)
            df = add_stochastic_tv(df, 14, 3, 3)
            df = add_stoch_rsi_tv(df, 14, 14, 3, 3)

            signal = check_signal(df)

            if signal == "LONG":
                long_agree.append(tf)

            if signal == "SHORT":
                short_agree.append(tf)

        except Exception as e:
            print(f"Error {symbol} {tf}: {e}")

    return long_agree, short_agree


In [11]:
def scan_market(pairs):
    results = {}

    for i, symbol in enumerate(pairs, 1):
        print(f"[{i}/{len(pairs)}] Scanning {symbol}...")

        long_agree, short_agree = analyze_symbol(symbol)

        # ===== LONG =====
        if len(long_agree) >= 2:
            print(f"🚀 LONG SIGNAL FOUND!")
            print(f"   Symbol: {symbol}")
            print(f"   Agreeing TFs ({len(long_agree)}): {', '.join(long_agree)}")
            print("-" * 40)

            results.setdefault(symbol, {})
            results[symbol]["LONG"] = long_agree

        # ===== SHORT =====
        if len(short_agree) >= 2:
            print(f"🔻 SHORT SIGNAL FOUND!")
            print(f"   Symbol: {symbol}")
            print(f"   Agreeing TFs ({len(short_agree)}): {', '.join(short_agree)}")
            print("-" * 40)

            results.setdefault(symbol, {})
            results[symbol]["SHORT"] = short_agree

    return results


In [12]:
signals = scan_market(filtered_pairs)

print("\n=== FINAL SUMMARY ===")
for symbol, sides in signals.items():
    for side, tfs in sides.items():
        print(f"{symbol} → {side} → {', '.join(tfs)}")


[1/271] Scanning BTC/USDT:USDT...
🚀 LONG SIGNAL FOUND!
   Symbol: BTC/USDT:USDT
   Agreeing TFs (3): 1h, 4h, 1d
----------------------------------------
[2/271] Scanning ETH/USDT:USDT...
🚀 LONG SIGNAL FOUND!
   Symbol: ETH/USDT:USDT
   Agreeing TFs (2): 4h, 1d
----------------------------------------
[3/271] Scanning BCH/USDT:USDT...
🚀 LONG SIGNAL FOUND!
   Symbol: BCH/USDT:USDT
   Agreeing TFs (3): 15m, 1h, 1d
----------------------------------------
[4/271] Scanning XRP/USDT:USDT...
🚀 LONG SIGNAL FOUND!
   Symbol: XRP/USDT:USDT
   Agreeing TFs (3): 1h, 4h, 1d
----------------------------------------
[5/271] Scanning LTC/USDT:USDT...
🚀 LONG SIGNAL FOUND!
   Symbol: LTC/USDT:USDT
   Agreeing TFs (2): 1h, 1d
----------------------------------------
[6/271] Scanning TRX/USDT:USDT...
[7/271] Scanning ETC/USDT:USDT...
[8/271] Scanning LINK/USDT:USDT...
🚀 LONG SIGNAL FOUND!
   Symbol: LINK/USDT:USDT
   Agreeing TFs (2): 1h, 1d
----------------------------------------
[9/271] Scanning XLM/US